# VM Resource Planner Walkthrough

Notebook này thực thi pipeline `vm_resource_planner.py` theo từng bước để quan sát rõ Phase 1→2→3 khi chuyển kết quả forecast thành kế hoạch VM allocation.

Các bước chính:
1. Load cấu hình & các mô hình forecast mới nhất
2. Sinh forecast trên tập test
3. Chuyển forecast thành nhu cầu CPU/RAM cần offload khỏi máy hiện tại
4. Giải bài toán phân bổ VM bằng Linear Programming cho hai kịch bản
5. Lập lịch VM theo từng time-bucket và lưu kết quả



In [25]:
import json
import importlib
from pathlib import Path

import pandas as pd

# Reload module to ensure latest changes are loaded
import vm_resource_planner
importlib.reload(vm_resource_planner)

from vm_resource_planner import (
    load_vm_catalog,
    load_latest_models,
    generate_forecasts,
    convert_forecasts_to_requirements,
    inverse_transform_forecasts,
    load_normalization_stats,
    summarize_peak_plans,
    build_schedule,
    HOST_SPEC,
    VM_TYPES_FILE,
    RESULTS_DIR,
    SUPPORTED_MODELS,
    TARGETS,
)

MODEL_NAME = "random_forest"  # thay đổi sang 'random_forest', 'svr', 'arimax' nếu muốn
RESULTS_DIR.mkdir(exist_ok=True)
print("✓ Libraries & planner helpers loaded")
print("Available models:", list(SUPPORTED_MODELS.keys()))
print("Current model:", MODEL_NAME)


✓ Libraries & planner helpers loaded
Available models: ['hybrid_prophet_lstm', 'arimax', 'random_forest', 'svr']
Current model: random_forest


In [26]:
vm_catalog = load_vm_catalog(VM_TYPES_FILE)
model_paths = load_latest_models(MODEL_NAME)

print("VM catalog:")
for spec in vm_catalog:
    print(f"  • {spec['name']}: {spec['vcpus']} vCPUs, {spec['memory_gb']} GB, ${spec['cost_per_hour']}/h")

print("\nLatest model checkpoints:")
for target, path in model_paths.items():
    print(f"  {target}: {path}")

HOST_SPEC


Latest model: E:\PROJECTS\Demand-Forecasting-and-Resource-Optimization-on-Cloud-Infrastructure\models\random_forest_memory_usage_pct_20251111_221757.pkl
Latest model: E:\PROJECTS\Demand-Forecasting-and-Resource-Optimization-on-Cloud-Infrastructure\models\random_forest_cpu_total_usage_20251111_221930.pkl
Latest model: E:\PROJECTS\Demand-Forecasting-and-Resource-Optimization-on-Cloud-Infrastructure\models\random_forest_system_load_20251111_221944.pkl
VM catalog:
  • B2s: 2 vCPUs, 4 GB, $0.0416/h
  • D2s_v3: 2 vCPUs, 8 GB, $0.096/h
  • D8s_v3: 8 vCPUs, 64 GB, $0.384/h
  • D32s_v3: 32 vCPUs, 128 GB, $1.536/h

Latest model checkpoints:
  memory_usage_pct: E:\PROJECTS\Demand-Forecasting-and-Resource-Optimization-on-Cloud-Infrastructure\models\random_forest_memory_usage_pct_20251111_221757.pkl
  cpu_total_usage: E:\PROJECTS\Demand-Forecasting-and-Resource-Optimization-on-Cloud-Infrastructure\models\random_forest_cpu_total_usage_20251111_221930.pkl
  system_load: E:\PROJECTS\Demand-Forecasting

{'total_cpu_cores': 1,
 'total_memory_gb': 4,
 'cpu_threshold_pct': 70,
 'memory_threshold_pct': 75}

In [27]:
forecast_df = generate_forecasts(model_paths, MODEL_NAME)
forecast_df.head()


✓ Model loaded: E:\PROJECTS\Demand-Forecasting-and-Resource-Optimization-on-Cloud-Infrastructure\models\random_forest_memory_usage_pct_20251111_221757.pkl
  Model: random_forest
  Target: memory_usage_pct
  Saved at: 2025-11-11 22:17:57
✓ Model loaded: E:\PROJECTS\Demand-Forecasting-and-Resource-Optimization-on-Cloud-Infrastructure\models\random_forest_cpu_total_usage_20251111_221930.pkl
  Model: random_forest
  Target: cpu_total_usage
  Saved at: 2025-11-11 22:19:30
✓ Model loaded: E:\PROJECTS\Demand-Forecasting-and-Resource-Optimization-on-Cloud-Infrastructure\models\random_forest_system_load_20251111_221944.pkl
  Model: random_forest
  Target: system_load
  Saved at: 2025-11-11 22:19:44
✓ Applied inverse transformation to forecasts (original scale)


,timestamp,memory_usage_pct,cpu_total_usage,system_load
0,2024-01-24 19:39:30,7.078415,0.113205,0.152157
1,2024-01-24 19:40:00,6.838327,0.067310,0.152157
2,2024-01-24 19:40:30,6.729628,0.072795,0.131861
3,2024-01-24 19:41:00,6.807487,0.061135,0.099938
4,2024-01-24 19:41:30,6.866732,0.071270,0.099938


In [28]:
# Hiển thị thống kê forecast để xác nhận inverse transformation hoạt động đúng
print("=== Forecast Statistics (Original Scale) ===")
for target in TARGETS:
    if target in forecast_df.columns:
        vals = forecast_df[target]
        print(f"  {target}: min={vals.min():.4f}, max={vals.max():.4f}, mean={vals.mean():.4f}")

# So sánh với normalization stats
norm_stats = load_normalization_stats()
print("\n=== Expected Original Stats (from cleaned_data.csv) ===")
for target in TARGETS:
    if target in norm_stats:
        stats = norm_stats[target]
        print(f"  {target}: mean={stats['mean']:.4f}, std={stats['std']:.4f}, min={stats['min']:.4f}, max={stats['max']:.4f}")


=== Forecast Statistics (Original Scale) ===
  memory_usage_pct: min=6.1214, max=8.5681, mean=6.8991
  cpu_total_usage: min=0.0436, max=0.9326, mean=0.0959
  system_load: min=0.0003, max=2.0977, mean=0.1223

=== Expected Original Stats (from cleaned_data.csv) ===
  memory_usage_pct: mean=6.4258, std=1.0366, min=4.2311, max=14.4301
  cpu_total_usage: mean=0.1049, std=0.0985, min=0.0365, max=1.6910
  system_load: mean=0.1297, std=0.2090, min=0.0000, max=5.0600


In [33]:
requirements_df = convert_forecasts_to_requirements(forecast_df, HOST_SPEC)
requirements_df[['timestamp','cpu_total_usage','cpu_required_cores','cpu_overflow_cores','memory_usage_pct','memory_required_gb','memory_overflow_gb']]


,timestamp,cpu_total_usage,cpu_required_cores,cpu_overflow_cores,memory_usage_pct,memory_required_gb,memory_overflow_gb
0,2024-01-24 19:39:30,0.113205,0.152157,0.0,7.078415,0.283137,0.0
1,2024-01-24 19:40:00,0.067310,0.152157,0.0,6.838327,0.273533,0.0
2,2024-01-24 19:40:30,0.072795,0.131861,0.0,6.729628,0.269185,0.0
3,2024-01-24 19:41:00,0.061135,0.099938,0.0,6.807487,0.272299,0.0
4,2024-01-24 19:41:30,0.071270,0.099938,0.0,6.866732,0.274669,0.0
...,...,...,...,...,...,...,...
17145,2024-01-30 18:32:00,0.187646,0.187646,0.0,7.349305,0.293972,0.0
17146,2024-01-30 18:32:30,0.170795,0.170795,0.0,7.413351,0.296534,0.0
17147,2024-01-30 18:33:00,0.179835,0.179835,0.0,7.277104,0.291084,0.0
17148,2024-01-30 18:33:30,0.245921,0.245921,0.0,7.000678,0.280027,0.0


In [30]:
peak_summary = summarize_peak_plans(requirements_df, vm_catalog)
peak_summary


{'peak_cpu_overflow': 1.3976890263497774,
 'peak_memory_overflow': 0.0,
 'minimize_overload_plan': {'allocation': {'B2s': 1},
  'total_vms': 1,
  'total_cpu': 2.0,
  'total_memory': 4.0,
  'total_cost': 0.0416},
 'minimize_cost_plan': {'allocation': {'B2s': 1},
  'total_vms': 1,
  'total_cpu': 2.0,
  'total_memory': 4.0,
  'total_cost': 0.0416}}

In [31]:
schedule_df = build_schedule(requirements_df, vm_catalog)
schedule_df.head()


,timestamp,cpu_overflow_cores,memory_overflow_gb,min_overload_plan,min_overload_cost_per_hour,min_cost_plan,min_cost_cost_per_hour
0,2024-01-24 19:30:00,0.000000,0.0,Host only,0.0000,Host only,0.0000
1,2024-01-24 20:00:00,0.415999,0.0,B2s×1,0.0416,B2s×1,0.0416
2,2024-01-24 20:30:00,0.000000,0.0,Host only,0.0000,Host only,0.0000
3,2024-01-24 21:00:00,0.000000,0.0,Host only,0.0000,Host only,0.0000
4,2024-01-24 21:30:00,0.844663,0.0,B2s×1,0.0416,B2s×1,0.0416


In [32]:
report_path = RESULTS_DIR / "vm_resource_planning_notebook.json"
schedule_path = RESULTS_DIR / "vm_schedule_notebook.csv"

schedule_records = schedule_df.copy()
schedule_records['timestamp'] = schedule_records['timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S')

report_payload = {
    'model': 'vm_resource_planner_notebook',
    'host_spec': HOST_SPEC,
    'peak_summary': peak_summary,
    'schedule': schedule_records.to_dict(orient='records'),
}

with open(report_path, 'w') as f:
    json.dump(report_payload, f, indent=2)

schedule_df.to_csv(schedule_path, index=False)

report_path, schedule_path


(WindowsPath('E:/PROJECTS/Demand-Forecasting-and-Resource-Optimization-on-Cloud-Infrastructure/forecast_result/vm_resource_planning_notebook.json'),
 WindowsPath('E:/PROJECTS/Demand-Forecasting-and-Resource-Optimization-on-Cloud-Infrastructure/forecast_result/vm_schedule_notebook.csv'))